# nano-relation-extractor

Runs the same stages as `uv run nano-re all`, one cell at a time, so
intermediate results can be inspected. All logic lives in the `nano_re`
package; these cells only orchestrate it.

Everything stays on this machine. Nothing is uploaded and no credentials are
needed: the corpora and the encoder are public downloads.

## Setup

Run once in a terminal:

```bash
uv sync
uv run jupyter lab
```

In [ ]:
from dataclasses import replace

from nano_re.config import PipelineConfig
from nano_re.pipeline import Pipeline

config = PipelineConfig.from_env()
config = config.with_overrides(
    data=replace(config.data, languages=("pl", "en"), limit=200),
    training=replace(config.training, epochs=1),
)
pipeline = Pipeline(config)

print("Encoder: ", config.model.backbone_name)
print("Languages:", ", ".join(config.data.languages))
print("Corpora:  ", config.data.relation_corpus, config.data.entity_corpus,
      config.data.english_relation_corpus)
print("Artifacts:", config.artifacts_dir.resolve())

## Data

Downloads the corpora, interleaves them by weight and derives the label schema
from what was actually read. The relation inventory depends on which languages
are in scope, so it is counted rather than declared.

In [ ]:
schema = pipeline.prepare()

print("Entity types:", ", ".join(schema.entity_types))
print("BIO tags:    ", schema.num_bio_labels)
print("Relations:   ", schema.num_relation_labels)

counts = pipeline.data_module.inventory.counts
top = sorted(counts.items(), key=lambda item: -item[1])[:10]
for relation_id, count in top:
    print(f"  {relation_id:<8} {count:>6}  {schema.describe_relation(relation_id)}")

In [ ]:
bundle = pipeline.data_module.build_corpus(config.data.train_split, training=True)
print(bundle.describe())

for index in range(len(bundle.dataset)):
    sample = bundle.dataset[index]
    if sample is None:
        continue
    print(f"\nFirst usable document: {sample.doc_id}")
    print(f"  sub-words {sample.input_ids.shape[0]}, entities {sample.num_entities},"
          f" candidate pairs {sample.num_pairs}")
    print(f"  relation supervision: {sample.has_relation_supervision}")
    print(f"  mention mask rows sum to one: "
          f"{[round(float(x), 3) for x in sample.mention_mask.sum(-1)[:4]]}")
    break

## Training

Both heads share one encoder and are trained under
`L = alpha * L_NER + beta * L_RE`. A corpus that annotates entities but not
relations is masked out of the relation term.

In [ ]:
training_report = pipeline.train()

best = training_report.best_evaluation
print(f"Best epoch:          {training_report.best_epoch}")
print(f"NER micro F1:        {best.ner.f1:.4f}")
print(f"Relation micro F1:   {best.relation.f1:.4f}")
print(f"Recall ceiling:      {best.relation_recall_ceiling:.4f}")

## Export and quantisation

The exporter compares the graph against PyTorch on three differently shaped
batches and fails if the relative deviation exceeds tolerance or if the two
would ever choose different classes.

In [ ]:
artifacts = pipeline.export()

print("Backend:            ", artifacts.export.exporter)
print("Dynamic shapes:     ", artifacts.export.dynamic_shapes_verified)
print(f"Relative deviation:  {artifacts.export.max_relative_deviation:.2e}")
print("Decisions match:    ", artifacts.export.decisions_match)
print(f"Size: {artifacts.quantization.source_bytes / 1e6:.1f} MB -> "
      f"{artifacts.quantization.target_bytes / 1e6:.1f} MB "
      f"({artifacts.quantization.compression_ratio:.2f}x)")

In [ ]:
benchmark = pipeline.benchmark(measure_accuracy=True)

print(f"FP32: {benchmark.fp32.median_ms:6.2f} ms/page  {benchmark.fp32.size_mb:7.1f} MB")
print(f"INT8: {benchmark.int8.median_ms:6.2f} ms/page  {benchmark.int8.size_mb:7.1f} MB")
print(f"Speedup {benchmark.speedup:.2f}x, size reduction {benchmark.size_reduction:.1%}")
print(f"F1 change from quantisation: NER {benchmark.ner_f1_delta:+.4f}, "
      f"relation {benchmark.relation_f1_delta:+.4f}")

## Bundle

Writes the model card from the measurements above, inventories the directory and
checks that nothing expected is missing.

In [ ]:
report = pipeline.package(
    training=training_report,
    benchmark=benchmark,
    quantization=artifacts.quantization,
)
print(report.render())
print("\nComplete:", report.is_complete)

## Using the model

Text of any length works: it is split into overlapping windows and the results
are merged. Structured identifiers are matched by rule and verified by checksum,
alongside whatever the model predicts.

In [ ]:
from nano_re.inference import RelationExtractor

extractor = RelationExtractor.from_bundle(
    pipeline.artifacts_dir, backend="onnx-int8", config=config
)
print("Backend:", extractor.backend_name, "\n")

text = (
    "Skai TV is a Greek free-to-air television network based in Piraeus. "
    "Skai TV is part of Skai Group, one of the largest media groups in the country."
)
print(extractor.extract(text).render())

In [ ]:
print((pipeline.artifacts_dir / "MODEL_CARD.md").read_text(encoding="utf-8"))